# Seasonal Agriculture Performance Analysis

## Problem Statement
Agricultural performance varies across seasons because of environmental conditions, farming practices, resource availability and market conditions. This project analyzes agricultural data to identify seasonal patterns, trends, relationships and variations in yield, production, profitability, resource usage and disease/pest risk.

## Objectives
- Explore and understand the dataset.
- Clean missing and duplicate records.
- Compare agricultural performance across seasons.
- Compare crop and regional performance.
- Analyze environmental and resource relationships.
- Evaluate water-use efficiency.
- Identify important observed correlations.
- Produce evidence-based findings and recommendations.


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

# Notebook is stored in notebooks/, so the dataset is one level above.
DATA_PATH = Path("../data/seasonal_agriculture_performance_dataset.csv")
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())


## 1. Dataset Understanding

In [ ]:
print("Columns:")
for column in df.columns:
    print("-", column)

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nUnique values:")
display(df.nunique().to_frame("unique_values"))


In [ ]:
print("Missing values:")
display(df.isna().sum().to_frame("missing_values"))

print("Duplicate rows:", df.duplicated().sum())

display(df.describe().T)


## 2. Data Cleaning

In [ ]:
df_clean = df.drop_duplicates().copy()

numeric_columns = df_clean.select_dtypes(include=np.number).columns

for column in numeric_columns:
    df_clean[column] = pd.to_numeric(df_clean[column], errors="coerce")

# Median imputation for numeric missing values.
missing_before = df_clean.isna().sum()
for column in numeric_columns:
    if df_clean[column].isna().any():
        df_clean[column] = df_clean[column].fillna(df_clean[column].median())

print("Original rows:", len(df))
print("Cleaned rows:", len(df_clean))
print("\nMissing values after cleaning:")
display(df_clean.isna().sum().to_frame("remaining_missing"))


## 3. Exploratory Data Analysis

In [ ]:
seasonal = df_clean.groupby("Season").agg(
    Farms=("Farm_ID", "count"),
    Avg_Yield=("Yield_Tonnes_Ha", "mean"),
    Total_Production=("Production_Tonnes", "sum"),
    Avg_Profit=("Profit_INR", "mean"),
    Avg_Water_Used=("Water_Used_m3", "mean"),
    Avg_Water_Efficiency=("Water_Efficiency_t_per_1000m3", "mean"),
    Avg_Risk=("Disease_Pest_Risk_pct", "mean"),
    Avg_Rainfall=("Rainfall_mm", "mean")
).sort_values("Avg_Yield", ascending=False)

display(seasonal.round(2))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
seasonal["Avg_Yield"].plot(kind="bar", ax=ax)
ax.set_title("Average Yield by Season")
ax.set_xlabel("Season")
ax.set_ylabel("Yield (tonnes/ha)")
plt.tight_layout()
plt.show()


In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 5))
x = np.arange(len(seasonal))
width = 0.36

ax1.bar(x - width/2, seasonal["Avg_Yield"], width, label="Average Yield")
ax2 = ax1.twinx()
ax2.plot(x, seasonal["Avg_Profit"]/1000, marker="o", linewidth=2, label="Average Profit")

ax1.set_xticks(x)
ax1.set_xticklabels(seasonal.index)
ax1.set_ylabel("Yield (tonnes/ha)")
ax2.set_ylabel("Profit (₹000)")
ax1.set_title("Seasonal Yield and Profit")

l1, a1 = ax1.get_legend_handles_labels()
l2, a2 = ax2.get_legend_handles_labels()
ax1.legend(l1 + l2, a1 + a2, loc="upper left")

plt.tight_layout()
plt.show()


## 4. Crop Analysis

In [ ]:
crop = df_clean.groupby("Crop").agg(
    Farms=("Farm_ID", "count"),
    Avg_Yield=("Yield_Tonnes_Ha", "mean"),
    Avg_Profit=("Profit_INR", "mean"),
    Avg_Water_Efficiency=("Water_Efficiency_t_per_1000m3", "mean")
).sort_values("Avg_Profit", ascending=False)

display(crop.round(2))


In [ ]:
plt.figure(figsize=(9, 5))
crop["Avg_Profit"].sort_values().plot(kind="barh")
plt.title("Average Profit by Crop")
plt.xlabel("Average Profit (INR)")
plt.tight_layout()
plt.show()


## 5. Regional Analysis

In [ ]:
state = df_clean.groupby("State").agg(
    Farms=("Farm_ID", "count"),
    Avg_Yield=("Yield_Tonnes_Ha", "mean"),
    Avg_Profit=("Profit_INR", "mean"),
    Avg_Water_Efficiency=("Water_Efficiency_t_per_1000m3", "mean"),
    Avg_Risk=("Disease_Pest_Risk_pct", "mean")
).sort_values("Avg_Yield", ascending=False)

display(state.round(2))


In [ ]:
plt.figure(figsize=(9, 5))
state["Avg_Yield"].sort_values().plot(kind="barh")
plt.title("Average Yield by State")
plt.xlabel("Average Yield (tonnes/ha)")
plt.tight_layout()
plt.show()


## 6. Environmental Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.scatterplot(data=df_clean, x="Rainfall_mm", y="Yield_Tonnes_Ha", ax=axes[0])
axes[0].set_title("Rainfall vs Yield")

sns.scatterplot(data=df_clean, x="Avg_Temperature_C", y="Yield_Tonnes_Ha", ax=axes[1])
axes[1].set_title("Temperature vs Yield")

plt.tight_layout()
plt.show()


## 7. Resource and Water Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.scatterplot(data=df_clean, x="Water_Used_m3", y="Production_Tonnes", ax=axes[0])
axes[0].set_title("Water Used vs Production")

sns.scatterplot(data=df_clean, x="Fertilizer_kg_ha", y="Yield_Tonnes_Ha", ax=axes[1])
axes[1].set_title("Fertilizer vs Yield")

plt.tight_layout()
plt.show()


In [ ]:
water = df_clean.groupby("Season")["Water_Efficiency_t_per_1000m3"].mean().sort_values(ascending=False)

display(water.to_frame("Average_Water_Efficiency").round(2))

water.plot(kind="bar", figsize=(8, 5), title="Water Efficiency by Season")
plt.ylabel("Tonnes per 1,000 m³")
plt.tight_layout()
plt.show()


## 8. Correlation Analysis

In [ ]:
numeric_df = df_clean.select_dtypes(include=np.number)
corr = numeric_df.corr()

plt.figure(figsize=(13, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()


In [ ]:
yield_corr = corr["Yield_Tonnes_Ha"].drop("Yield_Tonnes_Ha").sort_values(ascending=False)
profit_corr = corr["Profit_INR"].drop("Profit_INR").sort_values(ascending=False)

print("Top observed associations with yield:")
display(yield_corr.head(7).to_frame("correlation"))

print("Top observed associations with profit:")
display(profit_corr.head(7).to_frame("correlation"))


## 9. Regional Consistency Across Seasons

This comparison helps determine whether seasonal performance patterns are similar across states.


In [ ]:
state_season = df_clean.groupby(["State", "Season"])["Yield_Tonnes_Ha"].mean().unstack()
display(state_season.round(2))

plt.figure(figsize=(10, 6))
sns.heatmap(state_season, annot=True, fmt=".2f", cmap="YlGnBu")
plt.title("Average Yield by State and Season")
plt.tight_layout()
plt.show()


## 10. Irrigation Method Analysis

In [ ]:
irrigation = df_clean.groupby("Irrigation_Method").agg(
    Avg_Yield=("Yield_Tonnes_Ha", "mean"),
    Avg_Profit=("Profit_INR", "mean"),
    Avg_Water_Efficiency=("Water_Efficiency_t_per_1000m3", "mean")
).sort_values("Avg_Yield", ascending=False)

display(irrigation.round(2))

irrigation["Avg_Yield"].plot(kind="bar", figsize=(9,5), title="Average Yield by Irrigation Method")
plt.ylabel("Yield (tonnes/ha)")
plt.tight_layout()
plt.show()


## 11. Key Findings

Write the final findings using the numerical results generated above.

Recommended reporting points:
1. Best and worst performing seasons.
2. Highest and lowest profitability crops.
3. Highest and lowest performing regions.
4. Seasonal water-efficiency differences.
5. Important environmental/resource associations.
6. Whether seasonal patterns are consistent across regions.

**Interpretation rule:** correlation indicates association; it does not prove causation.


In [ ]:
best_season = seasonal["Avg_Yield"].idxmax()
best_crop = crop["Avg_Profit"].idxmax()
best_state = state["Avg_Yield"].idxmax()
best_water_season = water.idxmax()

print(f"Best average-yield season: {best_season}")
print(f"Highest average-profit crop: {best_crop}")
print(f"Highest average-yield state: {best_state}")
print(f"Best water-efficiency season: {best_water_season}")


## 12. Recommendations

- Use seasonal yield and profit comparisons for crop planning.
- Improve water management in lower-efficiency conditions.
- Investigate lower-performing regions using soil, irrigation and climate variables.
- Combine environmental, production and economic indicators for decisions.
- Add historical weather, soil and market-price data in future versions.
- Build an interactive BI dashboard.
- Consider predictive modeling after establishing a reliable baseline.


## 13. Conclusion

The analysis provides a structured view of agricultural performance across seasons, crops and regions. It combines data cleaning, exploratory analysis, visualization and correlation analysis to identify useful patterns in yield, production, profitability, resource use and risk. The findings provide a foundation for evidence-based agricultural planning and future predictive analytics.
